<img src="../../shared/alchemi-banner-left.png" alt="NVIDIA ALCHEMI: AI for Chemistry and Materials Science" style="display:block;box-sizing:border-box;width:100%;max-width:100%;height:auto;">

<style>
h1 {
  font-size: 2.35rem !important;
  line-height: 1.15 !important;
  margin: 0.7rem 0 1rem !important;
}
h2 {
  font-size: 1.9rem !important;
  line-height: 1.2 !important;
  margin: 0.6rem 0 1rem !important;
}
h3 {
  font-size: 1.35rem !important;
  line-height: 1.3 !important;
  margin: 1.9rem 0 0.75rem !important;
}
h4 {
  font-size: 1.08rem !important;
  line-height: 1.35 !important;
  margin: 1.35rem 0 0.65rem !important;
}
.alchemi-api-label,
span[aria-label="ALCHEMI Toolkit API"] {
  color: currentColor !important;
  font-size: 0.70rem !important;
  font-weight: 700;
  letter-spacing: 0.05em;
}
@supports (color: color-mix(in srgb, black, white)) {
  .alchemi-api-label,
  span[aria-label="ALCHEMI Toolkit API"] {
    color: color-mix(in srgb, currentColor 68%, #76B900 32%) !important;
  }
}
</style>

# ALCHEMI Core · Module 3: Compose and scale

**Goal:** Compose AIMNet2, D3 dispersion, and predicted-charge electrostatics; compare their interaction energies with published references; then carry the composed model through staged, inflight, and distributed execution.

[← Module 2 · Models and simulation](alchemi-core-02-models-and-simulation.ipynb) · **Module 3 of 3**


## From built-in models to model composition

Module 2 used built-in model wrappers to evaluate molecules and run simulations. Here we build one model from its terms: AIMNet2 supplies learned energy and atomic charges, D3 adds dispersion, and finite electrostatics reads those charges. Toolkit combines the terms behind the same model interface used in Module 2.

Ammonia, propyne, and phenol reappear inside published noncovalent complexes. Their interaction curves show what each term contributes before we carry the complete model into staged dynamics, inflight refill, and distributed execution.

In [1]:
import warnings

import torch
from helpers import core as helpers
from nvalchemi.data import AtomicData, Batch, InMemoryDataset
from nvalchemi.distributed import DistributedManager, DomainConfig, DomainParallel
from nvalchemi.dynamics import (
    FIRE2,
    BaseDynamics,
    ConvergenceHook,
    FusedStage,
    HostMemory,
    NVTLangevin,
    SizeAwareSampler,
)
from nvalchemi.models import (
    AIMNet2Wrapper,
    DFTD3ModelWrapper,
    PipelineGroup,
    PipelineModelWrapper,
    PipelineStep,
)
from nvalchemi.neighbors import compute_neighbors

warnings.filterwarnings(
    "ignore",
    message="Could not initialize using ENV, SLURM or OPENMPI methods.*",
    module="physicsnemo.distributed.manager",
)


In [2]:
helpers.configure_tutorial()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    device_label = (
        f"cuda:{torch.cuda.current_device()} · {torch.cuda.get_device_name(device)}"
    )
else:
    device_label = "cpu"
print(f"Compute device: {device_label}")

Compute device: cuda:0 · NVIDIA RTX 4000 SFF Ada Generation


In [3]:
checkpoint = helpers.model_checkpoint()
with helpers.suppress_known_aimnet_load_warning():
    model = AIMNet2Wrapper.from_checkpoint(
        checkpoint, device=device, compile_model=False
    ).eval()
helpers.freeze_model(model)
model.set_config("active_outputs", {"energy", "charges"})

<hr class="alchemi-section-divider" aria-hidden="true" style="border:0;border-top:1px solid #D6D9D4;margin:2.4rem 0 1rem;">

## 1 · Compose an interaction model

Module 2 evaluated one model wrapper. Here we build three versions of the same interaction model and run each one on published NCI Atlas dissociation scans.

### Load three frozen dissociation scans

The checked [NCI Atlas subset](data/nci_atlas/README.md) contains three complexes at ten relative separations. $R/R_e = 1$ is the equilibrium separation; smaller values compress the complex and larger values pull the fragments apart.

Every point contains the dimer `AB` and the frozen fragments `A` and `B`, giving 90 structures. We evaluate each structure once and calculate

$$E_{\mathrm{int}} = E(AB) - E(A) - E(B).$$

NCI Atlas supplies CCSD(T)/CBS interaction energies at all ten points. It also supplies ωB97M-D3(BJ)/def2-TZVPPD total energies for the same `AB`, `A`, and `B` geometries; their difference gives the DFT reference curve.

In [4]:
nci_atoms, nci_rows, nci_references = helpers.load_nci_atlas()

In [5]:
for row in helpers.nci_equilibrium_references(nci_references):
    print(
        f"{row['system_name']} ({row['interaction_class']}): "
        f"DFT-D3 {row['dft_d3_kcal_mol']:.3f}, "
        f"CCSD(T)/CBS {row['ccsd_t_cbs_kcal_mol']:.3f} kcal/mol"
    )

phenol - N-methylacetamide (neutral hydrogen bond): DFT-D3 -11.869, CCSD(T)/CBS -11.889 kcal/mol
propyne - methyl azide (dispersion-dominated): DFT-D3 -3.380, CCSD(T)/CBS -3.312 kcal/mol
ammonia - benzoate (ionic hydrogen bond): DFT-D3 -10.199, CCSD(T)/CBS -10.237 kcal/mol


In [ ]:
VIEW_SYSTEM = "phenol - N-methylacetamide"
view_complex = helpers.select_nci_equilibrium_complex(
    nci_atoms, nci_rows, VIEW_SYSTEM
)
helpers.show_molecule(view_complex, height=320)

In [7]:
# Convert each ASE structure into one Toolkit graph on the compute device.
nci_graphs = [
    AtomicData.from_atoms(structure, device=device)
    for structure in nci_atoms
]

In [ ]:
nci_batch = Batch.from_data_list(nci_graphs, device=device)
print(f"Atomic systems: {nci_batch.num_graphs} = 30 AB/A/B triplets")
print(f"Atoms:          {nci_batch.num_nodes:,}")
print(f"Device:         {nci_batch.device}")

In [ ]:
# ωB97M-D3(BJ) damping; a2 is in Bohr and cutoff is in Å.
d3 = DFTD3ModelWrapper(
    a1=0.566,
    a2=3.128,
    s8=0.3908,
    cutoff=15.0,
    param_file=helpers.d3_parameter_file(),
).to(device).eval()
d3.set_config("active_outputs", {"energy"})

In [10]:
# Use the same 15 Å neighbor radius for finite electrostatics.
coulomb = helpers.DirectCoulombAdapter(cutoff=15.0).to(device).eval()
coulomb.set_config("active_outputs", {"energy"})

This checkpoint uses three terms:

- AIMNet2 supplies a learned energy and predicted atomic charges;
- D3 computes dispersion from the shown damping constants and verified element tables;
- finite electrostatics reads the predicted charges and computes their pair energy.

The complete wrapper evaluates the terms through one Toolkit model interface.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `PipelineModelWrapper(groups=[PipelineGroup(...)], neighbor_adaptation="always")`
>
> **Input:** model steps, their data dependencies, and derivative groups  
> **Result:** one Toolkit model wrapper with a combined configuration and output mapping

### Build three model variants

We will call the same batch with:

1. `model`: AIMNet2 base;
2. `base_d3_model`: AIMNet2 base + D3;
3. `interaction_model`: AIMNet2 base + D3 + electrostatics.

D3 runs independently. Electrostatics consumes AIMNet2 charges, so `PipelineStep` wires `charges` to `partial_charges`.

In [ ]:
base_d3_model = PipelineModelWrapper(
    groups=[
        PipelineGroup(steps=[model], use_autograd=False),
        PipelineGroup(steps=[d3], use_autograd=False),
    ],
    neighbor_adaptation="always",
).to(device).eval()
base_d3_model.set_config("active_outputs", {"energy"})

In [ ]:
interaction_model = PipelineModelWrapper(
    groups=[
        PipelineGroup(
            steps=[
                PipelineStep(model, wire={"charges": "partial_charges"}),
                coulomb,
            ],
            use_autograd=True,
        ),
        PipelineGroup(steps=[d3], use_autograd=False),
    ],
    neighbor_adaptation="always",
).to(device).eval()
interaction_model.set_config("active_outputs", {"energy", "charges"})

In [13]:
# compute_neighbors mutates the clone with AIMNet-specific neighbor tensors.
aimnet_batch = nci_batch.clone()
compute_neighbors(aimnet_batch, config=model.model_config.neighbor_config)
with torch.no_grad():
    aimnet_outputs = model(aimnet_batch)

In [ ]:
base_d3_batch = nci_batch.clone()
compute_neighbors(
    base_d3_batch,
    config=base_d3_model.model_config.neighbor_config,
)
with torch.no_grad():
    base_d3_outputs = base_d3_model(base_d3_batch)

In [ ]:
interaction_batch = nci_batch.clone()
compute_neighbors(
    interaction_batch,
    config=interaction_model.model_config.neighbor_config,
)
with torch.no_grad():
    interaction_outputs = interaction_model(interaction_batch)

All three calls evaluate the same 90 structures. Each output contains one energy per structure; the complete model also returns one predicted charge per atom.

In [ ]:
charge_error = helpers.max_system_charge_error(
    nci_batch, interaction_outputs["charges"]
)

In [ ]:
{
    "AIMNet base": tuple(aimnet_outputs["energy"].shape),
    "base + D3": tuple(base_d3_outputs["energy"].shape),
    "base + D3 + electrostatics": tuple(interaction_outputs["energy"].shape),
    "max charge-sum error [e]": charge_error,
}

In [ ]:
curves = helpers.interaction_components(
    nci_rows,
    {
        "aimnet_base": aimnet_outputs["energy"],
        "base_d3": base_d3_outputs["energy"],
        "complete": interaction_outputs["energy"],
    },
)

In [ ]:
curve_summary = helpers.summarize_nci_interaction_curves(
    curves, nci_references
)

At equilibrium, the D3 shift is `base + D3` minus the base. The electrostatic shift is the complete model minus `base + D3`. Negative values make the interaction more attractive. The MAEs use all ten separation points.

In [ ]:
for row in curve_summary:
    print(
        f"{row['system']:30} | D3 {row['d3_shift']:+7.3f} | "
        f"electrostatics {row['electrostatic_shift']:+7.3f} | "
        f"MAE DFT/CC {row['dft_mae']:.3f}/{row['cc_mae']:.3f} kcal/mol"
    )

In [ ]:
helpers.plot_nci_interaction_curves(curves, nci_references)

phenol - N-methylacetamide     | D3  -3.227 | electrostatics +21.552 | MAE DFT/CC 0.285/0.304 kcal/mol
propyne - methyl azide         | D3  -2.379 | electrostatics  -2.256 | MAE DFT/CC 0.336/0.318 kcal/mol
ammonia - benzoate             | D3  -1.329 | electrostatics +23.443 | MAE DFT/CC 0.473/0.406 kcal/mol


<details>
<summary>Read the curves</summary>

At equilibrium, D3 shifts propyne–methyl azide by about −2.38 kcal/mol. Adding electrostatics shifts it by another −2.26 kcal/mol. For ammonia–benzoate, D3 contributes about −1.33 kcal/mol and electrostatics shifts the complete model by about +23.44 kcal/mol.

These shifts describe this checkpoint construction on the frozen NCI Atlas geometries.

</details>

## 2 · Batch a broader interaction survey

The three curves show how the energy terms change with separation. We now use one equilibrium geometry from each complex to measure the same model construction across a broader panel.

[NCIA250](https://github.com/Honza-R/NCIAtlas/tree/1816bfc72609d7deb1d4f93ab9e27eb13bb44bec/NCIA250) contains 250 representative neutral complexes selected from five NCI Atlas collections. The checkpoint supports 205 of them. The other 45 contain noble gases outside its element list. Dataset statistics retain all 250; model evaluation uses the supported 205.

Each interaction energy needs three inputs: AB, A, and B. The evaluation therefore contains 615 unequal atomic systems. The full compatible NCI Atlas contains 16,674 geometries and 50,022 AB/A/B graphs. NCIA250 keeps this live lesson compact.

In [ ]:
survey_atoms, survey_rows, survey_references, survey_inventory, survey_stats = (
    helpers.load_ncia250()
)

In [ ]:
print(f"NCIA250 complexes:       {survey_stats['total_complexes']}")
print(f"Evaluated complexes:     {survey_stats['compatible_complexes']}")
print(f"Outside checkpoint:      {survey_stats['excluded_complexes']} "
      f"({', '.join(survey_stats['excluded_elements'])})")
print(f"Model inputs (AB/A/B):   {survey_stats['compatible_graphs']}")
print(f"Atom rows in the Batch:  {survey_stats['compatible_atom_rows']:,}")
print("Evaluated / total by source:")
for source, total in survey_stats["source_counts"].items():
    kept = survey_stats["compatible_source_counts"].get(source, 0)
    print(f"  {source:14} {kept:>2} / {total}")

In [ ]:
# Convert each ASE structure into one Toolkit graph on the compute device.
survey_graphs = [
    AtomicData.from_atoms(structure, device=device)
    for structure in survey_atoms
]
survey_batch = Batch.from_data_list(survey_graphs, device=device)

In [ ]:
survey_counts = survey_batch.num_nodes_per_graph
print(f"Batch graphs:       {survey_batch.num_graphs}")
print(f"Batch atoms:        {survey_batch.num_nodes:,}")
print(f"Atoms per graph:    {int(survey_counts.min())} / "
      f"{float(survey_counts.float().median()):.0f} / {int(survey_counts.max())} "
      "(min / median / max)")
print(f"Batch device:       {survey_batch.device}")

### Compare the three model constructions

Run the AIMNet base, base + D3, and complete model on one heterogeneous `Batch`. The comparison uses the same 205 complexes and CCSD(T)/CBS reference energies for every model.

In [ ]:
survey_aimnet_batch = survey_batch.clone()
compute_neighbors(
    survey_aimnet_batch,
    config=model.model_config.neighbor_config,
)
with torch.no_grad():
    survey_aimnet_outputs = model(survey_aimnet_batch)

In [ ]:
survey_base_d3_batch = survey_batch.clone()
compute_neighbors(
    survey_base_d3_batch,
    config=base_d3_model.model_config.neighbor_config,
)
with torch.no_grad():
    survey_base_d3_outputs = base_d3_model(survey_base_d3_batch)

In [ ]:
survey_complete_batch = survey_batch.clone()
compute_neighbors(
    survey_complete_batch,
    config=interaction_model.model_config.neighbor_config,
)
with torch.no_grad():
    survey_complete_outputs = interaction_model(survey_complete_batch)

In [ ]:
survey_curves = helpers.interaction_components(
    survey_rows,
    {
        "aimnet_base": survey_aimnet_outputs["energy"],
        "base_d3": survey_base_d3_outputs["energy"],
        "complete": survey_complete_outputs["energy"],
    },
)

In [ ]:
survey_accuracy = helpers.interaction_accuracy_summary(
    survey_curves, survey_references
)

In [ ]:
for row in survey_accuracy.itertuples(index=False):
    print(
        f"{row.component}: MAE {row.mae_kcal_mol:.3f} kcal/mol; "
        f"best for {row.best_for_complexes} complexes"
    )

In [ ]:
helpers.plot_ncia250_survey(survey_inventory, survey_accuracy)

<details>
<summary>Read the panel result</summary>

The AIMNet base has a 7.998 kcal/mol MAE. Base + D3 gives 8.516 kcal/mol. The complete model, which adds charge-dependent electrostatics, reaches 1.519 kcal/mol.

</details>

`best_for_complexes` counts the complexes for which each model construction has the smallest absolute error.

**Scope:** NCIA250 contributes one equilibrium geometry per neutral complex. The comparison uses NCI Atlas CCSD(T)/CBS benchmark energies. Checkpoint training overlap with this panel is unknown.

### Benchmark batched inference

Time 205 independent AB/A/B triplet calls against one 615-system `Batch` on CUDA; the CPU path uses 24 triplets. Both routes reuse prepared neighbors.

In [ ]:
survey_batching = helpers.benchmark_interaction_batching(
    survey_graphs,
    survey_rows,
    survey_batch,
    interaction_model,
    warmups=1,
    repeats=3,
)

In [ ]:
batch_check = survey_batching["correctness"]
print(
    f"Batch/serial agreement: {batch_check['max_energy_delta_ev']:.2e} eV energy, "
    f"{batch_check['max_force_delta_ev_per_angstrom']:.2e} eV/Å forces"
)

In [ ]:
helpers.plot_interaction_batching(survey_batching)

The figure uses results measured in this kernel. A different GPU will produce different times. The chemistry, structures, outputs, and prepared neighbor lists stay fixed, so the comparison isolates model-call shape.

**Continue:** The [official composable-model example](https://nvidia.github.io/nvalchemi-toolkit/examples/advanced/07_composable_model_composition.html) develops `PipelineStep`, `PipelineGroup`, and `PipelineModelWrapper`. The [NCI Atlas data note](data/nci_atlas/README.md) records the exact source, papers, license, and checksum.

The next section keeps this wrapper interface and turns to execution at scale. **Module 4: Training and fine-tuning is in development.**

<hr class="alchemi-section-divider" aria-hidden="true" style="border:0;border-top:1px solid #D6D9D4;margin:2.4rem 0 1rem;">

## 3 · Scale execution

The composed wrapper now drives workflows in which systems occupy different stages and finish at different times.

### Share one model across workflow stages

`FusedStage` keeps one active `Batch`, ordered stage algorithms, and one model wrapper on the same device. `Batch.status` selects the stage for each system:

- status `0`: `FIRE2` optimizes the system;
- status `1`: `NVTLangevin` advances its dynamics;
- status `2`: the configured stage chain is complete.

In the trace, A and C stay in FIRE2 while B enters NVT. One shared `interaction_model(batch)` call still evaluates A, B, and C; status masks choose which stage updates each system.

<details>
<summary>How one fused iteration is ordered</summary>

1. Each stage applies `pre_update` to its status mask.
2. The shared model evaluates the full active `Batch`.
3. Each stage applies `post_update` to the same mask.
4. Stage completion updates `status` for the next iteration.

</details>

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `fused = stage_a + stage_b`
>
> **Input:** compatible dynamics stages using one model wrapper and device path  
> **Result:** a `FusedStage` with stage codes `0` and `1`; its default `exit_status` is `2`

<img src="assets/fused-stage-flow-6be7a7c697638fd9.svg" alt="FusedStage applies an ordered optimization and dynamics chain. FIRE2 updates A and C, NVTLangevin updates B, and one shared interaction-model call evaluates A, B, and C before the active Batch enters its next iteration." style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0.8rem 0 0.45rem;">

In [ ]:
complexes = helpers.load_recurring_nci_complexes()
labels = tuple(complexes)
atoms = list(complexes.values())
example_batch = Batch.from_data_list(
    [AtomicData.from_atoms(structure, device=device) for structure in atoms],
    device=device,
)
example_counts = example_batch.num_nodes_per_graph.tolist()

In [ ]:
model.set_config("active_outputs", {"energy", "forces", "charges"})
d3.set_config("active_outputs", {"energy", "forces"})
interaction_model.set_config(
    "active_outputs", {"energy", "forces", "charges"}
)

In [ ]:
FUSED_FMAX = 0.30  # eV/Å; chosen to expose different stage transitions.
FIRE_MAX_UPDATES = 3
NVT_UPDATES = 4
# This Toolkit pin counts the status-entry iteration once.
ENTRY_COUNTER_OFFSET = 1

In [ ]:
fused_fire = FIRE2(
    model=interaction_model, dt=0.01, maxstep=0.04,
    n_steps=FIRE_MAX_UPDATES,
    convergence_hook=ConvergenceHook.from_fmax(FUSED_FMAX),
)
fused_nvt = NVTLangevin(
    model=interaction_model, dt=0.25, temperature=300.0, friction=0.1,
    random_seed=11, n_steps=NVT_UPDATES + ENTRY_COUNTER_OFFSET,
)

In [ ]:
fused = fused_fire + fused_nvt

# The composed wrapper rebuilds its shared neighbor source as positions change.
for hook in interaction_model.make_neighbor_hooks():
    fused.register_hook(hook)

status_trace = helpers.FusedStatusTrace()
fused.register_fused_hook(status_trace)

In [ ]:
for status, stage in fused.sub_stages:
    print(f"status {status}: {type(stage).__name__}")
print(f"exit status: {fused.exit_status}")

status 0: FIRE2
status 1: NVTLangevin
exit status: 2


In [ ]:
for letter, name, structure, count in zip(
    "ABC", labels, atoms, example_counts, strict=True
):
    charge = int(structure.info["charge"])
    formula = structure.get_chemical_formula()
    print(f"{letter}: {name:28} | {formula:10} | {count:2} atoms | charge {charge:+d}")

#### Try it: predict how the stages separate

> ✏️ **TRY IT**
>
> A, B, and C all start at status `0` in FIRE2. Before running the trace, predict which complex enters status `1`, NVT, first and which model calculation still uses all three systems together.
>
> **Success:** your prediction answers both questions before the trace reveals the stage changes.

The run mutates `fused_input` and records into `status_trace`. Rerun this section from the configuration cell to repeat it.

In [ ]:
fused_input = helpers.prepare_dynamics_batch(example_batch)
fused_result = fused.run(fused_input, n_steps=16)

In [ ]:
stage_names = {0: "FIRE2", 1: "NVT", 2: "done"}
for step, codes in status_trace.rows:
    states = "  ".join(
        f"{letter}:{stage_names[code]}" for letter, code in zip("ABC", codes, strict=True)
    )
    print(f"step {step}: {states}")

nvt_counts = [sum(codes[i] == 1 for _, codes in status_trace.rows) for i in range(3)]
print(f"NVT updates:      {nvt_counts}")
print(f"fused iterations: {fused.step_count}")
print(f"final status:     {fused_result.status.squeeze(-1).tolist()}")

step 0: A:FIRE2  B:FIRE2  C:FIRE2
step 1: A:FIRE2  B:FIRE2  C:NVT
step 2: A:FIRE2  B:FIRE2  C:NVT
step 3: A:NVT  B:NVT  C:NVT
step 4: A:NVT  B:NVT  C:NVT
step 5: A:NVT  B:NVT  C:done
step 6: A:NVT  B:NVT  C:done
NVT updates:      [4, 4, 4]
fused iterations: 7
final status:     [2, 2, 2]


> ✅ **SAMPLE SOLUTION**

<details>
<summary><strong>Open the sample solution</strong></summary>

B, ammonia-benzoate, enters NVT first. The shared composed-model call still evaluates A, B, and C together. `Batch.status` selects FIRE2 updates for A and C and NVT updates for B. A and C reach NVT later through the three-update FIRE2 limit.

</details>

### Refill the active `Batch` as systems finish

`FusedStage` moves systems between FIRE2 and NVT inside the active `Batch`. Inflight mode also removes completed systems, stores their results, and refills the available capacity:

- `HostMemory` stores completed systems on CPU;
- the active `Batch` keeps systems still running;
- `SizeAwareSampler` selects waiting systems that fit the atom and system limits.

The queue contains two copies of each 14, 18, and 25 atom complex. Three systems fill the first active `Batch`; three wait. The active capacity is three systems and 57 atoms.

This dataset stores structures before neighbor construction, so the compact example sets `max_edges=None` and uses atom and system limits.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `SizeAwareSampler(dataset, max_atoms=..., max_edges=..., max_batch_size=...)`
>
> **Input:** a dataset and active-`Batch` capacity limits  
> **Result:** a sampler that selects replacements within those limits

In [ ]:
queue_seed = helpers.prepare_dynamics_batch(example_batch).cpu()
seed_graphs = queue_seed.to_data_list()
queue_batch = Batch.from_data_list(
    [graph.clone() for _ in range(2) for graph in seed_graphs],
    device="cpu",
)

In [ ]:
queue_letters = tuple("ABCDEF")
for name, count in zip(labels, example_counts, strict=True):
    print(f"{name:28} | {count:2} atoms | 2 queued copies")

A: Ethyne     |  4 atoms
B: Acetamide  |  9 atoms
C: Isobutane  | 14 atoms
D: Ethyne     |  4 atoms
E: Acetamide  |  9 atoms
F: Isobutane  | 14 atoms


In [ ]:
queue = InMemoryDataset(in_memory_batch=queue_batch, device=device)
sampler = SizeAwareSampler(
    queue, max_atoms=queue_seed.num_nodes, max_edges=None,
    max_batch_size=queue_seed.num_graphs, shuffle=False,
)
results_sink = HostMemory(capacity=len(queue))

In [ ]:
print(f"queued systems:    {len(queue)}")
print(f"active system cap: {sampler.max_batch_size}")
print(f"active atom cap:   {sampler.max_atoms}")
print(f"stored edge cap:   {sampler.max_edges}")

In [ ]:
queue_fire = FIRE2(
    model=interaction_model, dt=0.01, maxstep=0.04,
    n_steps=FIRE_MAX_UPDATES,
    convergence_hook=ConvergenceHook.from_fmax(FUSED_FMAX),
)
queue_nvt = NVTLangevin(
    model=interaction_model, dt=0.25, temperature=300.0, friction=0.1,
    random_seed=23, n_steps=NVT_UPDATES + ENTRY_COUNTER_OFFSET,
)

In [ ]:
inflight = FusedStage(
    sub_stages=[(0, queue_fire), (1, queue_nvt)],
    sampler=sampler,
    sinks=[results_sink],
    refill_frequency=1,
)

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `FusedStage(sub_stages=..., sampler=..., sinks=..., refill_frequency=1)`
>
> **Input:** status-labeled workflow stages, a size-aware source of molecules, and result sinks.  
> **Result:** one active `Batch` whose finished molecules are replaced as capacity opens.

In [ ]:
# Rebuild the composed model's shared neighbor source as positions change.
for hook in interaction_model.make_neighbor_hooks():
    inflight.register_hook(hook)

# This fused hook prints every admitted system's stage before each step.
monitor = helpers.InflightStatusPrinter(
    results_sink,
    {0: "FIRE", 1: "NVT"},
    sampler=sampler,
    system_labels=dict(enumerate(queue_letters)),
)
inflight.register_fused_hook(monitor)

#### Try it: predict the first refill

> ✏️ **TRY IT**
>
> The first active `Batch` contains A, B, and C. D, E, and F are waiting. Predict which active system leaves first, which waiting system enters, and which systems remain active.
>
> **Success:** your prediction names one departing system, one entering system, and the systems that stay active.

This run consumes the sampler. Rerun this section from `queue_seed` through cleanup to repeat it.

In [ ]:
# batch=None asks the sampler to fill the first active Batch from the queue.
inflight_result = inflight.run(batch=None, n_steps=30)

step  active molecules               waiting  done  event
   0  A:FIRE B:FIRE C:FIRE                 3     0  initial fill
   1  A:FIRE B:FIRE C:NVT                  3     0  
   2  A:FIRE B:FIRE C:NVT                  3     0  
   3  A:NVT B:NVT C:NVT                    3     0  
   4  A:NVT B:NVT C:NVT                    3     0  
   5  A:NVT B:NVT D:FIRE                   2     1  refill +[D] | finished [C]
   6  A:NVT B:NVT D:NVT                    2     1  
   7  D:NVT E:FIRE F:FIRE                  0     3  refill +[E,F] | finished [A,B]
   8  D:NVT E:FIRE F:FIRE                  0     3  


   9  D:NVT E:FIRE F:FIRE                  0     3  
  10  E:NVT F:NVT                          0     4  finished [D]


  11  E:NVT F:NVT                          0     4  
  12  E:NVT F:NVT                          0     4  
  13  E:NVT F:NVT                          0     4  


<img src="assets/inflight-batching-flow-ab128daf945b0847.svg" alt="Inflight refill moves completed B to HostMemory and waiting D into the active Batch. A and C remain active; E and F remain queued." style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0.8rem 0 0.45rem;">

> ✅ **SAMPLE SOLUTION**

<details>
<summary><strong>Open the sample solution</strong></summary>

B completes first and moves to `HostMemory`. D enters the active `Batch` at FIRE2. A and C remain active. The monitor reports the event as `refill +[D] | finished [B]`.

</details>

In [ ]:
completed = results_sink.drain()
completed_ids = completed.system_id.squeeze(-1).tolist()
assert inflight_result is None and sampler.exhausted
assert sorted(completed_ids) == list(range(len(queue)))
assert torch.all(completed.status == inflight.exit_status)

print(f"completed systems: {completed.num_graphs} on {completed.device}")
print(f"atom counts:       {completed.num_nodes_per_graph.tolist()}")
print(f"charges:           {completed.charge.squeeze(-1).tolist()}")
print(f"system IDs:        {completed_ids}")
print(f"final status:      {completed.status.squeeze(-1).tolist()}")

completed systems: 6 on cpu
atom counts:       [14, 4, 9, 14, 9, 4]
system IDs:        [2, 0, 1, 3, 4, 5]
final status:      [2, 2, 2, 2, 2, 2]


In [ ]:
queue.close()

The monitor runs at `BEFORE_STEP`, so each row describes the active `Batch` about to enter one shared model evaluation.

- **active systems** pairs each admission letter with its current stage;
- **waiting** counts dataset samples available to the sampler;
- **done** counts completed results stored in `HostMemory`;
- **event** names systems entering and leaving the active `Batch`.

A letter can change from `FIRE` to `NVT` while remaining active. At the first refill, B leaves for `HostMemory` and D enters at `FIRE`.

With `refill_frequency=1`, Toolkit checks for completed systems after every fused step. The run ends after the sampler is exhausted and every active system completes the workflow.

`FusedStage` advances systems through stages on one device or rank. Inflight batching keeps new work entering the active `Batch`. `DistributedPipeline` places workflow stages on separate ranks and connects them with `|`.

**Continue:** See the official [multi-stage pipeline](https://nvidia.github.io/nvalchemi-toolkit/examples/intermediate/01_multistage_pipeline.html) and [inflight batching](https://nvidia.github.io/nvalchemi-toolkit/examples/intermediate/04_inflight_batching.html) examples for larger workloads and longer stage chains.

### Send independent systems through rank-owned stages

Toolkit usually starts one process per CUDA GPU for distributed execution. Each process has a **rank**.

`DistributedPipeline` maps each rank to one workflow stage. For `FIRE2 | NVTLangevin`, rank 0 runs FIRE2. When a system finishes FIRE2, its transfer `Batch` moves to rank 1 for NVT.

This notebook constructs the rank-to-stage map in one process. A complete run adds communication buffers, a first-stage sampler, a final-stage sink, and a matching two-rank launch.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `pipeline = stage_a | stage_b`
>
> **Input:** two workflow stages in execution order  
> **Result:** a `DistributedPipeline` with `stage_a` on rank 0 and `stage_b` on rank 1

In [ ]:
pipeline_preview = (
    FIRE2(model=model, dt=0.01, maxstep=0.04, n_steps=2)
    | NVTLangevin(
        model=model, dt=0.25, temperature=300.0,
        friction=0.1, random_seed=31, n_steps=2,
    )
)
for rank, stage in pipeline_preview.stages.items():
    print(f"rank {rank} owns {type(stage).__name__}")

rank 0 owns FIRE2
rank 1 owns NVE


### Split one large system across ranks

**Switching scaling pattern:** `DistributedPipeline` moves independent systems between ranks. `DomainParallel` partitions one large periodic system across ranks. This one-process preview places the neutral phenol-N-methylacetamide complex in a periodic control cell.

Follow the public sequence used in a distributed launch: initialize, configure, partition, run, gather, and clean up. This walkthrough keeps every atom on rank 0. A multi-process launch distributes spatial ownership across ranks.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> ```python
> with DomainParallel(dynamics=inner, config=config) as domain:
>     local = domain.partition(batch)
>     local = domain.run(local)
>     result = domain.gather(local, dst=0)
> ```
>
> **Input:** one global `Batch`, an inner dynamics object, and domain configuration  
> **Result:** rank-local spatial work plus a gathered global result on the destination rank

In [ ]:
domain_atoms = atoms[2].copy()
domain_atoms.set_cell((30.0, 30.0, 30.0))
domain_atoms.center()
domain_atoms.set_pbc(True)

In [ ]:
domain_graph = AtomicData.from_atoms(domain_atoms, device=device)
# Add empty energy and force fields for DomainParallel to update and gather.
domain_graph.add_system_property("energy", torch.zeros((1, 1), device=device))
domain_batch = Batch.from_data_list([domain_graph], device=device)
domain_batch.add_key(
    "forces", [torch.zeros_like(domain_graph.positions)],
    level="node", overwrite=True,
)

In [ ]:
# Match the model cutoff; zero skin uses that exact radius.
# mesh=None selects the single-rank path. grid_dims defaults to None,
# so SpatialPartitioner derives the spatial grid.
domain_config = DomainConfig(
    cutoff=float(model.model_config.neighbor_config.cutoff),
    skin=0.0,
    mesh=None,
    compile=False,
)

In [ ]:
DistributedManager.initialize()
manager = DistributedManager()
world_size = manager.world_size
DOMAIN_STEPS = 1  # One model step exercises the domain control path.
# BaseDynamics hosts model evaluation and neighbor hooks inside DomainParallel.
domain_evaluator = BaseDynamics(
    model=model, n_steps=DOMAIN_STEPS, hooks=model.make_neighbor_hooks()
)

In [ ]:
try:
    with DomainParallel(
        dynamics=domain_evaluator,
        config=domain_config,
        n_steps=DOMAIN_STEPS,
        device_type=device.type,
    ) as domain:
        # partition assigns owned atoms, run evaluates them, and gather rebuilds rank 0.
        owned_batch = domain.partition(domain_batch)
        domain_result = domain.run(owned_batch, n_steps=DOMAIN_STEPS)
        gathered_result = domain.gather(domain_result, dst=0)
finally:
    DistributedManager.cleanup()

In [ ]:
print("world size:       ", world_size)
print("input atoms:      ", domain_batch.num_nodes)
print("rank-0 owned atoms:", int(owned_batch.positions.shape[0]))
print("gathered atoms:   ", int(gathered_result.positions.shape[0]))
print("gathered type:    ", type(gathered_result).__name__)

**Scope:** World size one exercises initialization, `partition`, `run`, `gather`, and cleanup while rank 0 owns every atom. Multi-rank work adds a device mesh, rank-owned input, derived grid settings, and energy and force parity before timing.

**Continue:** The [official distributed simulations guide](https://nvidia.github.io/nvalchemi-toolkit/userguide/distributed.html) explains device meshes, partitioning, halo exchange, and launch requirements.

### Four-GPU execution shape

Rank 0 passes the global `Batch` to `partition(...)`; ranks 1–3 pass `None`. Every rank runs `domain.run(local_batch)`, then participates in `gather(..., dst=0)`.

| Call | Rank 0 | Ranks 1–3 |
|---|---|---|
| `partition(...)` | passes `domain_batch` | passes `None` |
| `run(...)` | evaluates its local spatial batch | evaluates its local spatial batch |
| `gather(..., dst=0)` | receives the global result | sends local results |

Halo exchange supplies neighbors across spatial boundaries during model evaluation. `SpatialPartitioner` derives the three-dimensional regions from the cell, cutoff, and rank count.

A measured multi-GPU run should report owned and halo atom counts on every rank.

### Choose the scaling pattern

`Batch` packs independent systems for a model call. The execution pattern determines whether work is divided by system, workflow stage, or spatial region.

| Workload | Execution pattern | How the work is divided |
|---|---|---|
| Many independent molecules run FIRE2 and NVT on one GPU and finish at different times | `FusedStage` with inflight batching | `status` moves each molecule between stages; the sampler fills capacity released by completed molecules |
| A steady stream of molecular batches runs FIRE2 on GPU 0, then moves to GPU 1 for NVT | `DistributedPipeline` | GPU 0 sends each completed FIRE2 batch to GPU 1 and starts the next batch while GPU 1 runs NVT |
| One periodic system is too large for one GPU | `DomainParallel` | Each rank owns a spatial region of the same system and exchanges halo atoms across region boundaries |

The FusedStage and inflight diagrams above show how per-system state and active capacity change. The official [distributed examples](https://nvidia.github.io/nvalchemi-toolkit/examples/distributed/index.html) show complete multi-process launches for `DistributedPipeline` and `DomainParallel`.

<hr class="alchemi-section-divider" aria-hidden="true" style="border:0;border-top:1px solid #D6D9D4;margin:2.4rem 0 1rem;">

## Module 3 recap

- `PipelineStep` wires AIMNet2 charges into finite electrostatics, while `PipelineGroup` keeps derivative ownership explicit.
- The 205-complex survey compares the composed model with published interaction energies and evaluates all 615 AB/A/B systems in one `Batch`.
- `FusedStage` uses `Batch.status` to advance systems through FIRE2 and NVT with one shared model call.
- `SizeAwareSampler` refills active capacity as `HostMemory` receives completed systems.
- `DistributedPipeline` assigns stages to ranks; `DomainParallel` partitions one large periodic system across ranks.

## Resources and acknowledgements

- [Toolkit user guides](https://nvidia.github.io/nvalchemi-toolkit/userguide/index.html)
- [Toolkit examples gallery](https://nvidia.github.io/nvalchemi-toolkit/examples/)

Thanks to the NVIDIA ALCHEMI team and the contributors who shaped this course, with special thanks to Kelvin Lee, Roman Zubatyuk, Ryan Reese, Justin Smith, Nikita Fedik, and Piero.

Questions or feedback: reach out to Justin Smith, Nikita Fedik, or Piero.

[← Module 2 · Models and simulation](alchemi-core-02-models-and-simulation.ipynb) · **End of the current Core path**
